# Nutrisense-AI — Complete ML Training Pipeline
## Capstone Project | African Leadership University

**Author:** Jean Jabo  
**Dataset:** Food-101 + KenyanFood13 (ViT, 114 classes) · Rwanda DHS 2019-20 + WHO STEPS 2012 (XGBoost)

| Part | Task |
|------|------|
| 0 | Setup & Dataset URL Verification |
| 1 | Food Image Classifier (ViT fine-tuning on Food-101 + KenyanFood13, 114 classes) |
| 2 | Disease Risk Models (XGBoost — Anemia · Diabetes · Overweight) |
| 3 | HuggingFace Deployment |

**Before running:**
1. Right panel → Add Data → Competition Datasets → `pytorch-opencv-course-classification` (KenyanFood13)
2. Right panel → Add Data → Public Datasets → search **`dansbecker/food-101`** and add it (Food-101, 101 classes)
3. Right panel → Add Data → Your Datasets → `nutrisense-dhs-rwanda` (Rwanda DHS 2019-20)
4. Right panel → Add Data → Your Datasets → `rwanda-steps-2012` (WHO STEPS 2012)
5. Session options → Accelerator → **GPU T4 x2**
6. Session options → Internet → **On**
7. Add-ons → Secrets → add `HF_TOKEN` with your HuggingFace write token

In [ ]:
%matplotlib inline
!pip install timm shap xgboost huggingface_hub -q 2>/dev/null

import os, warnings, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image, UnidentifiedImageError

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.amp import autocast, GradScaler
import timm

import shap, joblib
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = '/kaggle/working'
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}  |  GPUs: {torch.cuda.device_count()}')
print(f'Device  : {DEVICE}')

## Part 0 — Dataset URL Verification

In [ ]:
DATASET_URLS = {
    'KenyanFood13 (Kaggle Competition)' : 'https://www.kaggle.com/c/pytorch-opencv-course-classification',
    'Food-101 (Kaggle — dansbecker)'    : 'https://www.kaggle.com/datasets/dansbecker/food-101',
    'WHO STEPS Rwanda 2012-2013'        : 'https://extranet.who.int/ncdsmicrodata/index.php/catalog/709',
    'Rwanda DHS 2019-20'                : 'https://dhsprogram.com/data/dataset/Rwanda_Standard-DHS_2019.cfm',
    'Roboflow Cassava Dataset'          : 'https://universe.roboflow.com/cc-aryuc/cassava-qbdwf',
    'Roboflow Amaranth Dataset'         : 'https://universe.roboflow.com/unkraut/unkraut',
    'Rwanda NISR EICV Portal'           : 'https://microdata.statistics.gov.rw/index.php/catalog/119',
}

print('=' * 62)
print('DATASET URL VERIFICATION')
print('=' * 62)
rows = []
for name, url in DATASET_URLS.items():
    try:
        r = requests.head(url, timeout=10, allow_redirects=True)
        ok     = r.status_code < 400
        status = f'OK {r.status_code}'
    except Exception as e:
        ok     = False
        status = 'unreachable'
    rows.append({'Dataset': name, 'Status': status, 'Available': ok})
    print(f'  {status:15s} {name}')

df_urls = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.axis('off')
tbl = ax.table(
    cellText=df_urls[['Dataset','Status']].values,
    colLabels=['Dataset', 'HTTP Status'],
    cellLoc='left', loc='center'
)
tbl.auto_set_font_size(False); tbl.set_fontsize(10)
tbl.auto_set_column_width([0, 1])
plt.title('Dataset Availability Check', fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/00_url_verification.png', dpi=120, bbox_inches='tight')
plt.show()

## Part 1 — Food Image Classifier
### 1.1 Dataset Exploration (KenyanFood13)

In [ ]:
COMP_DIR   = '/kaggle/input/competitions/pytorch-opencv-course-classification'
TRAIN_CSV  = f'{COMP_DIR}/train.csv'
IMAGES_DIR = f'{COMP_DIR}/images/images'

df_train = pd.read_csv(TRAIN_CSV)
print('train.csv columns:', df_train.columns.tolist())
print(df_train.head(3).to_string())
print(f'\nShape: {df_train.shape}')

# Auto-detect image filename column and label column
id_col    = next((c for c in df_train.columns
                  if any(k in c.lower() for k in ['image','file','id','name'])),
                 df_train.columns[0])
label_col = next((c for c in df_train.columns
                  if any(k in c.lower() for k in ['label','class','category','target'])),
                 df_train.columns[1])
print(f'\nUsing: id_col={id_col!r}  label_col={label_col!r}')

classes      = sorted(df_train[label_col].unique().tolist())
NUM_CLASSES  = len(classes)
cls2idx      = {c: i for i, c in enumerate(classes)}
class_counts = df_train[label_col].value_counts().to_dict()

# Detect whether filenames already carry an extension
sample_fname = str(df_train[id_col].iloc[0])
HAS_EXT      = sample_fname.lower().endswith(('.jpg', '.jpeg', '.png'))

def img_path(fname):
    fname = str(fname)
    if not HAS_EXT and not fname.lower().endswith(('.jpg','.jpeg','.png')):
        fname = fname + '.jpg'
    return os.path.join(IMAGES_DIR, fname)

print(f'\nImages dir   : {IMAGES_DIR}')
print(f'KenyanFood13 classes : {NUM_CLASSES}')
print(f'Total images : {len(df_train):,}')
for c in sorted(class_counts, key=lambda x: -class_counts[x]):
    print(f'  {c:25s}  {class_counts[c]:>4} images')

# ── Food-101 (dansbecker/food-101 — stored as zip, extract first) ────────────
FOOD101_ZIP      = '/kaggle/input/food-101/food-101.zip'
FOOD101_EXTRACT  = '/kaggle/working/food101'
FOOD101_DIR      = f'{FOOD101_EXTRACT}/food-101/images'

if os.path.isfile(FOOD101_ZIP) and not os.path.isdir(FOOD101_DIR):
    print('\nExtracting food-101.zip (10 GB — takes ~5 min) ...')
    import zipfile
    with zipfile.ZipFile(FOOD101_ZIP, 'r') as zf:
        zf.extractall(FOOD101_EXTRACT)
    print('Extraction complete.')

f101_records = []
if os.path.isdir(FOOD101_DIR):
    for cls_dir in sorted(Path(FOOD101_DIR).iterdir()):
        if cls_dir.is_dir():
            for img_f in cls_dir.glob('*.jpg'):
                f101_records.append({'filepath': str(img_f), 'label': cls_dir.name})
    df_food101 = pd.DataFrame(f101_records)
    print(f'\nFood-101 : {len(df_food101):,} images across {df_food101["label"].nunique()} classes')
else:
    df_food101 = pd.DataFrame(columns=['filepath', 'label'])
    print('\nFood-101 not found — will train on KenyanFood13 only')

In [ ]:
# EDA 1 — Class distribution bar chart
sorted_cls = sorted(class_counts, key=lambda x: -class_counts[x])
counts     = [class_counts[c] for c in sorted_cls]
palette    = sns.color_palette('viridis', NUM_CLASSES)

fig, ax = plt.subplots(figsize=(13, 5))
bars = ax.bar(sorted_cls, counts, color=palette, edgecolor='white', linewidth=0.8)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(cnt), ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Food Class', fontsize=12)
ax.set_ylabel('Number of Images', fontsize=12)
ax.set_title('KenyanFood13 — Images per Class', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/01_class_distribution.png', dpi=120)
plt.show()
print('Saved → 01_class_distribution.png')

In [ ]:
# EDA 2 — Sample image grid (one per class, from CSV)
n_cols = 5
n_rows = (NUM_CLASSES + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.2))
axes = axes.flatten()
for i, cls in enumerate(classes):
    row   = df_train[df_train[label_col] == cls].iloc[0]
    fpath = img_path(row[id_col])
    try:
        img = Image.open(fpath).convert('RGB')
        axes[i].imshow(img)
    except Exception as e:
        axes[i].text(0.5, 0.5, f'missing\n{e}', ha='center', va='center',
                     fontsize=7, transform=axes[i].transAxes)
    axes[i].set_title(str(cls).replace('_', ' ').title(), fontsize=9, fontweight='bold')
    axes[i].axis('off')
for j in range(i + 1, len(axes)):
    axes[j].axis('off')
plt.suptitle('KenyanFood13 — One Sample per Class', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/02_sample_images.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → 02_sample_images.png')

In [ ]:
# EDA 3 — Image size distribution (sample from CSV)
widths, heights = [], []
for _, row in df_train.sample(min(500, len(df_train)), random_state=42).iterrows():
    try:
        w, h = Image.open(img_path(row[id_col])).size
        widths.append(w); heights.append(h)
    except: pass

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, vals, label, color in zip(
    axes,
    [widths, heights],
    ['Width (px)', 'Height (px)'],
    ['#4C72B0', '#55A868']
):
    ax.hist(vals, bins=35, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.median(vals), color='red', linestyle='--',
               label=f'Median: {int(np.median(vals))}px')
    ax.axvline(224, color='orange', linestyle=':', label='Target: 224px')
    ax.set_xlabel(label); ax.legend()
    ax.set_title(f'Image {label.split()[0]} Distribution', fontweight='bold')
plt.suptitle('KenyanFood13 — Image Dimensions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03_image_sizes.png', dpi=120)
plt.show()
print('Saved → 03_image_sizes.png')

In [ ]:
# EDA 4 — Augmentation visualization
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
IMG_SIZE      = 224

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def denorm(t):
    m = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    s = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (t * s + m).clamp(0,1).permute(1,2,0).numpy()

sample_cls  = classes[0]
sample_row  = df_train[df_train[label_col] == sample_cls].iloc[0]
orig        = Image.open(img_path(sample_row[id_col])).convert('RGB')

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes[0,0].imshow(orig.resize((224,224))); axes[0,0].set_title('Original', fontweight='bold')
for idx in range(1, 10):
    axes[idx//5][idx%5].imshow(denorm(train_tf(orig)))
    axes[idx//5][idx%5].set_title(f'Aug #{idx}', fontsize=9)
for ax in axes.flatten(): ax.axis('off')
plt.suptitle(f'Data Augmentation Pipeline — {sample_cls}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_augmentation.png', dpi=120)
plt.show()
print('Saved → 04_augmentation.png')

### 1.2 Data Splits

In [ ]:
BATCH_SIZE   = 32
ACCUM_STEPS  = 4
EPOCHS       = 8
LR           = 1e-4
WEIGHT_DECAY = 0.05
NUM_WORKERS  = 2
MODEL_NAME   = 'vit_base_patch16_224.orig_in21k'
CKPT_PATH    = f'{SAVE_DIR}/food_vit_best.pth'
CLASSES_TXT  = f'{SAVE_DIR}/class_names.txt'

# ── Build unified class list (KenyanFood13 + Food-101) ───────────────────────
df_kenyan_unified = pd.DataFrame({
    'filepath': df_train[id_col].apply(img_path),
    'label':    df_train[label_col].astype(str),
})
combined_df = pd.concat([df_kenyan_unified, df_food101], ignore_index=True)
classes     = sorted(combined_df['label'].unique().tolist())
NUM_CLASSES = len(classes)
cls2idx     = {c: i for i, c in enumerate(classes)}

print(f'Combined dataset : {len(combined_df):,} images | {NUM_CLASSES} classes')
print(f'  KenyanFood13   : {len(df_kenyan_unified):,} images  ({df_kenyan_unified["label"].nunique()} classes)')
print(f'  Food-101       : {len(df_food101):,} images  ({df_food101["label"].nunique()} classes)')

with open(CLASSES_TXT, 'w') as f:
    f.write('\n'.join(classes))

# ── Dataset that works with direct filepaths (handles both sources) ───────────
class UnifiedDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records   = records   # list of (filepath_str, label_idx)
        self.transform = transform
    def __len__(self):
        return len(self.records)
    def __getitem__(self, i):
        fpath, label = self.records[i]
        try:
            image = Image.open(fpath).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224))
        if self.transform:
            image = self.transform(image)
        return image, label

# Build (filepath, label_idx) list and stratified splits
all_records = [(row.filepath, cls2idx[row.label]) for row in combined_df.itertuples()]
all_labels  = [r[1] for r in all_records]

from sklearn.model_selection import train_test_split as sk_split
tr_val_rec, test_rec, tr_val_lbl, _ = sk_split(
    all_records, all_labels, test_size=0.15, random_state=42, stratify=all_labels
)
tr_rec, val_rec = sk_split(
    tr_val_rec, test_size=round(0.15 / 0.85, 6), random_state=42, stratify=tr_val_lbl
)

train_ds = UnifiedDataset(tr_rec,  train_tf)
val_ds   = UnifiedDataset(val_rec, val_tf)
test_ds  = UnifiedDataset(test_rec, val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,   shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

n_train, n_val, n_test = len(train_ds), len(val_ds), len(test_ds)

# Split visualization
fig, ax = plt.subplots(figsize=(8, 4))
splits  = {'Train\n70%': n_train, 'Val\n15%': n_val, 'Test\n15%': n_test}
bars    = ax.bar(splits.keys(), splits.values(),
                 color=['#4C72B0', '#55A868', '#C44E52'], edgecolor='white', linewidth=1.5)
for bar, v in zip(bars, splits.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 80,
            f'{v:,}', ha='center', fontweight='bold', fontsize=12)
ax.set_title(f'Train / Val / Test Split — {NUM_CLASSES} classes', fontsize=13, fontweight='bold')
ax.set_ylabel('Images')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/05_data_split.png', dpi=120)
plt.show()
print(f'Train {n_train:,}  Val {n_val:,}  Test {n_test:,}  |  {NUM_CLASSES} classes')

### 1.3 ViT Fine-tuning

In [ ]:
model     = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, epochs=EPOCHS,
    steps_per_epoch=max(1, len(train_loader) // ACCUM_STEPS)
)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler  = GradScaler(device='cuda')

print(f'Model  : {MODEL_NAME}')
print(f'Params : {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'Epochs : {EPOCHS}  |  Effective batch: {BATCH_SIZE * ACCUM_STEPS}')

def topk_acc(out, tgt, k=1):
    with torch.no_grad():
        _, pred = out.topk(k, 1); correct = pred.t().eq(tgt.view(1,-1).expand_as(pred.t()))
        return correct[:k].reshape(-1).float().mean().item() * 100

history  = {'train_loss': [], 'val_top1': [], 'val_top5': [], 'lr': []}
best_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train(); running_loss = 0.0; optimizer.zero_grad()
    for step, (imgs, lbls) in enumerate(tqdm(train_loader, desc=f'Ep {epoch}/{EPOCHS} train', leave=False)):
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        with autocast(device_type='cuda', dtype=torch.float16):
            loss = loss_fn(model(imgs), lbls) / ACCUM_STEPS
        scaler.scale(loss).backward()
        running_loss += loss.item() * ACCUM_STEPS
        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step(); optimizer.zero_grad()
    avg_loss = running_loss / len(train_loader)

    model.eval(); t1 = t5 = 0.0
    with torch.no_grad():
        for imgs, lbls in tqdm(val_loader, desc=f'Ep {epoch}/{EPOCHS} val', leave=False):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            with autocast(device_type='cuda', dtype=torch.float16):
                out = model(imgs)
            t1 += topk_acc(out, lbls, 1) * imgs.size(0)
            t5 += topk_acc(out, lbls, min(5, NUM_CLASSES)) * imgs.size(0)
    top1 = t1 / len(val_ds); top5 = t5 / len(val_ds)
    history['train_loss'].append(avg_loss)
    history['val_top1'].append(top1)
    history['val_top5'].append(top5)
    history['lr'].append(scheduler.get_last_lr()[0])
    if top1 > best_acc:
        best_acc = top1
        torch.save({'epoch': epoch, 'state': model.state_dict(), 'best_acc': best_acc}, CKPT_PATH)
    print(f'Ep {epoch:02d}/{EPOCHS}  loss {avg_loss:.4f}  top-1 {top1:.2f}%  top-5 {top5:.2f}%')

print(f'\nBest val top-1: {best_acc:.2f}%')

### 1.4 Evaluation

In [ ]:
# Learning curves
ep_range = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(ep_range, history['train_loss'], 'o-', color='#4C72B0', linewidth=2)
axes[0].set(title='Training Loss', xlabel='Epoch', ylabel='Cross-Entropy'); axes[0].grid(alpha=0.3)

axes[1].plot(ep_range, history['val_top1'], 'o-', color='#55A868', label='Top-1', linewidth=2)
axes[1].plot(ep_range, history['val_top5'], 's--', color='#C44E52', label='Top-5', linewidth=2)
axes[1].set(title='Validation Accuracy', xlabel='Epoch', ylabel='Accuracy (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep_range, history['lr'], 'o-', color='#8172B2', linewidth=2)
axes[2].set(title='Learning Rate Schedule', xlabel='Epoch', ylabel='LR')
axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.suptitle(f'ViT Training Curves — Food-101 + KenyanFood13 ({NUM_CLASSES} classes)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/06_learning_curves.png', dpi=120)
plt.show()
print('Saved -> 06_learning_curves.png')

In [ ]:
# Load best checkpoint and evaluate on test set
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['state'])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc='Test set evaluation'):
        imgs = imgs.to(DEVICE)
        with autocast(device_type='cuda', dtype=torch.float16):
            out = model(imgs)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(lbls.numpy())

test_top1 = (np.array(all_preds) == np.array(all_labels)).mean() * 100
print(f'Test Top-1 Accuracy: {test_top1:.2f}%')

# Confusion matrix — skip cell annotations when classes > 30 (unreadable at 114)
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
annotate = NUM_CLASSES <= 30
cell_size = max(0.18, 8.0 / NUM_CLASSES)
fig_size  = max(14, NUM_CLASSES * cell_size)

fig, axes = plt.subplots(1, 2, figsize=(fig_size, fig_size * 0.5))
tick_lbl  = classes if NUM_CLASSES <= 30 else []
for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    ['Confusion Matrix (Raw Counts)', 'Confusion Matrix (Normalized)'],
    ['d', '.2f']
):
    sns.heatmap(data, annot=annotate, fmt=fmt if annotate else '', cmap='Blues',
                xticklabels=tick_lbl, yticklabels=tick_lbl, ax=ax,
                annot_kws={'size': 7})
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_xlabel('Predicted', fontsize=10); ax.set_ylabel('True', fontsize=10)
    if tick_lbl:
        ax.tick_params(axis='x', rotation=45)

plt.suptitle(f'ViT Test Set Confusion Matrix  (Top-1: {test_top1:.1f}%  |  {NUM_CLASSES} classes)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/07_confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved -> 07_confusion_matrix.png')

In [ ]:
# Per-class precision / recall / F1
report    = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
df_report = pd.DataFrame(report).T.drop(['accuracy','macro avg','weighted avg'])

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
for ax, metric, cmap in zip(axes, ['precision','recall','f1-score'], ['Blues','Greens','Oranges']):
    data = df_report[[metric]].sort_values(metric, ascending=False)
    sns.heatmap(data, annot=True, fmt='.3f', cmap=cmap, vmin=0, vmax=1, ax=ax,
                cbar_kws={'shrink': 0.8})
    ax.set_title(metric.upper(), fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Per-Class Classification Metrics — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/08_per_class_metrics.png', dpi=120)
plt.show()
print('Saved → 08_per_class_metrics.png')
print('\n' + classification_report(all_labels, all_preds, target_names=classes))

In [ ]:
# Sample correct vs incorrect predictions
model.eval()
correct_ex, wrong_ex = [], []
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs = imgs.to(DEVICE)
        with autocast(device_type='cuda', dtype=torch.float16):
            out = model(imgs)
        preds = out.argmax(1).cpu()
        for i in range(len(lbls)):
            img_np = denorm(imgs[i].cpu())
            item   = (img_np, lbls[i].item(), preds[i].item())
            if lbls[i] == preds[i] and len(correct_ex) < 6: correct_ex.append(item)
            if lbls[i] != preds[i] and len(wrong_ex)   < 6: wrong_ex.append(item)
        if len(correct_ex) >= 6 and len(wrong_ex) >= 6: break

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
for i, (img, true, pred) in enumerate(correct_ex):
    axes[0,i].imshow(img)
    axes[0,i].set_title(f'✅ {classes[true]}', fontsize=8, color='green')
    axes[0,i].axis('off')
for i, (img, true, pred) in enumerate(wrong_ex):
    axes[1,i].imshow(img)
    axes[1,i].set_title(f'❌ True:{classes[true]}\nPred:{classes[pred]}', fontsize=7, color='red')
    axes[1,i].axis('off')

plt.suptitle('Sample Predictions — Correct (top) vs Incorrect (bottom)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/09_sample_predictions.png', dpi=120)
plt.show()
print('Saved → 09_sample_predictions.png')

## Part 2 — Disease Risk Models (XGBoost)
### 2.1 Data Loading

**Dual-source strategy:**

| Dataset | Rows | Anemia label | Diabetes label | Overweight label |
|---------|------|-------------|----------------|-----------------|
| Rwanda DHS 2019-20 | ~14 600 | ✅ Real haemoglobin (women) / calibrated (men) | ⚠️ Logistic proxy | ✅ Real BMI (women) / calibrated (men) |
| WHO STEPS 2012 | ~5 000 | ⚠️ Synthesised at Rwanda rates | ✅ **Real fasting glucose ≥ 7.0 mmol/L** | ✅ **Real measured BMI ≥ 25** |
| **Combined** | **~19 600** | DHS real + STEPS synthesised | **STEPS real + DHS proxy** | **Both real** |

`DATA_SOURCE = 'combined'` merges both. Set to `'dhs'` or `'steps'` to train on one source only.  
Fallback: synthetic Rwanda-calibrated data if neither Kaggle dataset is attached.

In [ ]:
FEATURES = [
    'age', 'sex',
    'energy_kcal', 'protein_g', 'fat_g', 'carb_g',
    'iron_mg', 'vitC_mg', 'vitA_mcg',
    'fiber_g', 'sugar_g', 'calcium_mg', 'zinc_mg', 'sodium_mg',
]
TARGETS = ['anemia', 'diabetes', 'overweight']

# ── Data source ───────────────────────────────────────────────────────────────
# 'combined' = DHS (real anemia) + STEPS (real diabetes/overweight) — recommended
# 'dhs'      = DHS only (diabetes label is a logistic proxy)
# 'steps'    = STEPS only (anemia label is synthesised at Rwanda rates)
DATA_SOURCE = 'combined'

DHS_PATH   = '/kaggle/input/nutrisense-dhs-rwanda/nutrisense_dhs.csv'
STEPS_PATH = '/kaggle/input/rwanda-steps-2012/nutrisense_steps.csv'

def generate_rwanda_synthetic(n=15000):
    """
    Synthetic fallback calibrated to Rwanda/East Africa population statistics.
    Used only when neither Kaggle dataset is attached.
      Anemia     ~22% women / ~8% men   (Rwanda DHS 2019-20 report)
      Diabetes   ~3.4%                  (Rwanda STEPS 2012-13 report)
      Overweight ~22%                   (Rwanda STEPS 2012-13 report)
    """
    rng    = np.random.default_rng(42)
    sex    = rng.integers(1, 3, n)
    age    = rng.uniform(18, 65, n)
    energy = rng.normal(1850, 550, n).clip(600, 4000)
    protein= rng.normal(55,  20, n).clip(10, 150)
    fat    = rng.normal(55,  20, n).clip(10, 150)
    carb   = rng.normal(280, 80, n).clip(50, 600)
    iron   = rng.normal(10,   5, n).clip(1,  35)
    vitC   = rng.normal(70,  40, n).clip(0,  300)
    vitA   = rng.normal(550,250, n).clip(50, 2500)
    fiber  = rng.normal(22,   8, n).clip(3,  60)
    sugar  = rng.normal(60,  30, n).clip(5,  200)
    calc   = rng.normal(650,200, n).clip(100,2000)
    zinc   = rng.normal(8,    3, n).clip(1,  30)
    sodium = rng.normal(2500,800, n).clip(400,6000)
    hgb    = np.where(sex==2, 12.5, 14.0) + iron*0.07 - age*0.015 + rng.normal(0, 0.9, n)
    bmi    = 22.0 + (energy-1850)/700 + age*0.04 - protein*0.025 + rng.normal(0, 3.5, n)
    hba1c  = 5.1 + (sugar-60)/90 + (bmi-22.0)*0.07 + rng.normal(0, 0.65, n)
    df = pd.DataFrame({
        'age': age, 'sex': sex, 'energy_kcal': energy, 'protein_g': protein,
        'fat_g': fat, 'carb_g': carb, 'iron_mg': iron, 'vitC_mg': vitC,
        'vitA_mcg': vitA, 'fiber_g': fiber, 'sugar_g': sugar,
        'calcium_mg': calc, 'zinc_mg': zinc, 'sodium_mg': sodium,
        '_hgb': hgb, '_bmi': bmi, '_hba1c': hba1c,
    })
    df['anemia']     = ((df['sex']==2)&(df['_hgb']<12.0) | (df['sex']==1)&(df['_hgb']<13.0)).astype(int)
    df['overweight'] = (df['_bmi'] >= 25.0).astype(int)
    df['diabetes']   = (df['_hba1c'] >= 6.5).astype(int)
    return df

# ── Load datasets ─────────────────────────────────────────────────────────────

dhs_ok   = os.path.exists(DHS_PATH)
steps_ok = os.path.exists(STEPS_PATH)

print(f'DHS   : {"✅ found" if dhs_ok   else "❌ not found"}  {DHS_PATH}')
print(f'STEPS : {"✅ found" if steps_ok else "❌ not found"}  {STEPS_PATH}')
print()

if DATA_SOURCE == 'combined' and dhs_ok and steps_ok:
    dhs   = pd.read_csv(DHS_PATH)
    steps = pd.read_csv(STEPS_PATH)
    df    = pd.concat([dhs, steps], ignore_index=True)
    print(f'✅ Combined: DHS {len(dhs):,} rows + STEPS {len(steps):,} rows = {len(df):,} total')

elif DATA_SOURCE == 'dhs' and dhs_ok:
    df = pd.read_csv(DHS_PATH)
    print(f'✅ Loaded Rwanda DHS 2019-20: {df.shape[0]:,} rows')

elif DATA_SOURCE == 'steps' and steps_ok:
    df = pd.read_csv(STEPS_PATH)
    print(f'✅ Loaded WHO STEPS 2012: {df.shape[0]:,} rows')

elif dhs_ok:
    df = pd.read_csv(DHS_PATH)
    DATA_SOURCE = 'dhs'
    print(f'⚠️  STEPS not found — falling back to DHS only: {df.shape[0]:,} rows')

else:
    df = generate_rwanda_synthetic(15000)
    DATA_SOURCE = 'synthetic'
    print(f'⚠️  No datasets found — using synthetic fallback: {df.shape[0]:,} rows')

df = df.dropna(subset=FEATURES + TARGETS)
print(f'\nFinal  : {df.shape[0]:,} rows')
print(f'Anemia     {df.anemia.mean()*100:.1f}%')
print(f'Diabetes   {df.diabetes.mean()*100:.1f}%  {"← real FBG from STEPS" if "steps" in DATA_SOURCE else "← logistic proxy"}')
print(f'Overweight {df.overweight.mean()*100:.1f}%  {"← real BMI from STEPS" if "steps" in DATA_SOURCE else "← BMI proxy"}')

### 2.2 Exploratory Data Analysis

In [ ]:
# EDA — Feature distributions
key_feats = ['age','energy_kcal','iron_mg','protein_g','sugar_g','carb_g']
palette   = sns.color_palette('viridis', len(key_feats))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feat, color in zip(axes.flatten(), key_feats, palette):
    ax.hist(df[feat], bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[feat].mean(),   color='red',    linestyle='--', label=f'Mean {df[feat].mean():.1f}')
    ax.axvline(df[feat].median(), color='orange', linestyle=':',  label=f'Median {df[feat].median():.1f}')
    ax.set_title(feat.replace('_',' ').title(), fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle(f'Feature Distributions — {DATA_SOURCE.upper()} Data', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/10_feature_distributions.png', dpi=120)
plt.show()
print('Saved → 10_feature_distributions.png')

In [ ]:
# EDA — Correlation heatmap
corr = df[FEATURES].corr()
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, ax=ax, annot_kws={'size': 8}, square=True,
            xticklabels=[f.replace('_','\n') for f in FEATURES],
            yticklabels=[f.replace('_','\n') for f in FEATURES])
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/11_correlation_heatmap.png', dpi=120)
plt.show()
print('Saved → 11_correlation_heatmap.png')

In [ ]:
# EDA — Class imbalance (disease prevalence)
COLORS = {'anemia': '#4C72B0', 'diabetes': '#C44E52', 'overweight': '#55A868'}
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
for ax, tgt in zip(axes, TARGETS):
    counts = df[tgt].value_counts().sort_index()
    bars   = ax.bar(['Negative','Positive'], counts.values,
                    color=[COLORS[tgt], COLORS[tgt]],
                    edgecolor='white', linewidth=1.5)
    bars[0].set_alpha(0.35)
    bars[1].set_alpha(0.90)
    for bar, v in zip(bars, counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+80,
                f'{v:,}\n({v/len(df)*100:.1f}%)', ha='center', fontweight='bold', fontsize=10)
    ax.set_title(tgt.upper(), fontweight='bold', fontsize=13, color=COLORS[tgt])
    ax.set_ylabel('Count')

plt.suptitle('Disease Label Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/12_class_imbalance.png', dpi=120)
plt.show()
print('Saved → 12_class_imbalance.png')

### 2.3 XGBoost Training (5-fold CV)

In [ ]:
X = df[FEATURES].astype(float).values
Y = df[TARGETS].values
X_tr, X_te, Y_tr, Y_te = train_test_split(X, Y, test_size=0.2, random_state=42)

XGB_PARAMS = dict(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    eval_metric='auc', random_state=42, n_jobs=-1,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models_xgb = {}; explainers = {}; cv_results = {}

for i, tgt in enumerate(TARGETS):
    y_tr, y_te = Y_tr[:, i], Y_te[:, i]
    sw         = compute_sample_weight('balanced', y_tr)
    print(f'\n{"="*52}\n{tgt.upper()}  (train pos rate {y_tr.mean()*100:.1f}%)')

    make_pipe = lambda: Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    XGBClassifier(**XGB_PARAMS))
    ])

    aucs, aps = [], []
    for fold, (tr, va) in enumerate(cv.split(X_tr, y_tr)):
        p = make_pipe()
        p.fit(X_tr[tr], y_tr[tr], clf__sample_weight=sw[tr])
        prob = p.predict_proba(X_tr[va])[:, 1]
        aucs.append(roc_auc_score(y_tr[va], prob))
        aps.append(average_precision_score(y_tr[va], prob))
        print(f'  Fold {fold+1}: AUC {aucs[-1]:.3f}  AP {aps[-1]:.3f}')

    print(f'  CV  AUC {np.mean(aucs):.3f} ± {np.std(aucs):.3f}  |  AP {np.mean(aps):.3f}')

    pipe = make_pipe()
    pipe.fit(X_tr, y_tr, clf__sample_weight=sw)
    models_xgb[tgt] = pipe
    explainers[tgt] = shap.TreeExplainer(pipe.named_steps['clf'])
    cv_results[tgt] = {'roc_auc': float(np.mean(aucs)), 'avg_prec': float(np.mean(aps)),
                        'std': float(np.std(aucs))}

print('\n✅ All 3 models trained.')

### 2.4 Evaluation

In [ ]:
# ROC + Precision-Recall curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for tgt in TARGETS:
    y_te   = Y_te[:, TARGETS.index(tgt)]
    prob   = models_xgb[tgt].predict_proba(X_te)[:, 1]
    color  = COLORS[tgt]
    auc    = roc_auc_score(y_te, prob)
    ap     = average_precision_score(y_te, prob)
    fpr, tpr, _ = roc_curve(y_te, prob)
    prec, rec, _ = precision_recall_curve(y_te, prob)
    axes[0].plot(fpr, tpr, color=color, linewidth=2.5, label=f'{tgt.title()} (AUC={auc:.3f})')
    axes[1].plot(rec, prec, color=color, linewidth=2.5, label=f'{tgt.title()} (AP={ap:.3f})')

axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set(title='ROC Curves', xlabel='False Positive Rate', ylabel='True Positive Rate')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set(title='Precision-Recall Curves', xlabel='Recall', ylabel='Precision')
axes[1].legend(); axes[1].grid(alpha=0.3)
for ax in axes: ax.set_xlim(0,1); ax.set_ylim(0,1.02)

plt.suptitle('XGBoost Risk Models — Test Set Performance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/13_roc_pr_curves.png', dpi=120)
plt.show()
print('Saved → 13_roc_pr_curves.png')

In [ ]:
# Confusion matrices (one per disease)
THRESHOLDS = {'anemia': 0.40, 'diabetes': 0.35, 'overweight': 0.50}
fig, axes  = plt.subplots(1, 3, figsize=(16, 5))
for ax, tgt in zip(axes, TARGETS):
    y_te  = Y_te[:, TARGETS.index(tgt)]
    prob  = models_xgb[tgt].predict_proba(X_te)[:, 1]
    preds = (prob >= THRESHOLDS[tgt]).astype(int)
    cm    = confusion_matrix(y_te, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Neg','Pos'], yticklabels=['Neg','Pos'],
                annot_kws={'size': 14})
    ax.set_title(tgt.upper(), fontweight='bold', fontsize=12, color=COLORS[tgt])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/14_confusion_matrices.png', dpi=120)
plt.show()
print('Saved → 14_confusion_matrices.png')

In [ ]:
# SHAP feature importance (top 10 per disease)
scaler_fit = StandardScaler().fit(X_tr)
X_sample   = scaler_fit.transform(X_te[:300])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
shap_top  = {}
for ax, tgt in zip(axes, TARGETS):
    sv        = explainers[tgt].shap_values(X_sample)
    mean_shap = np.abs(sv).mean(0)
    order     = mean_shap.argsort()[::-1][:10]
    feat_lbls = [FEATURES[j].replace('_',' ') for j in order[::-1]]
    ax.barh(feat_lbls, mean_shap[order[::-1]], color=COLORS[tgt], alpha=0.85)
    ax.set_title(f'SHAP — {tgt.upper()}', fontweight='bold', color=COLORS[tgt])
    ax.set_xlabel('Mean |SHAP value|')
    shap_top[tgt] = [{'f': FEATURES[j], 'v': round(float(np.mean(sv[:,j])),3)} for j in order[:5]]

plt.suptitle('Feature Importance via SHAP (top 10)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/15_shap_importance.png', dpi=120)
plt.show()
print('Saved → 15_shap_importance.png')

In [ ]:
# SHAP waterfall — individual prediction explanation
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, tgt in zip(axes, TARGETS):
    sv    = explainers[tgt].shap_values(X_sample[[0]])
    vals  = sv[0]
    order = np.abs(vals).argsort()[::-1][:8]
    clrs  = [COLORS[tgt] if v > 0 else '#AAAAAA' for v in vals[order]]
    ax.barh([FEATURES[j].replace('_',' ') for j in order[::-1]],
             vals[order[::-1]], color=clrs[::-1], alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Waterfall — {tgt.upper()}\n(sample #0 explanation)',
                 fontweight='bold', color=COLORS[tgt])
    ax.set_xlabel('SHAP value')

plt.suptitle('Individual Prediction Explanation (SHAP Waterfall)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/16_shap_waterfall.png', dpi=120)
plt.show()
print('Saved → 16_shap_waterfall.png')

In [ ]:
# CV results summary table
fig, ax = plt.subplots(figsize=(9, 3))
ax.axis('off')
tbl_data = [[t.upper(),
             f"{cv_results[t]['roc_auc']:.3f}",
             f"± {cv_results[t]['std']:.3f}",
             f"{cv_results[t]['avg_prec']:.3f}"]
            for t in TARGETS]
tbl = ax.table(cellText=tbl_data,
               colLabels=['Disease', 'ROC-AUC', 'Std', 'Avg Precision'],
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(13); tbl.scale(1.5, 2.2)
plt.title('5-Fold Cross-Validation Results', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/17_cv_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → 17_cv_results.png')

In [ ]:
# Save XGBoost bundle
BUNDLE_PATH = f'{SAVE_DIR}/nutrisense_model.joblib'
bundle = {
    'features'   : FEATURES,
    'models'     : models_xgb,
    'explainers' : explainers,
    'shap_top'   : shap_top,
    'thresholds' : THRESHOLDS,
    'data_source': DATA_SOURCE,
    'label_definitions': {
        'anemia'    : 'Hb < 12 g/dL (women) / < 13 g/dL (men) [WHO 2011] — DHS real biomarker',
        'overweight': 'BMI >= 25 [WHO] — STEPS real measurement + DHS real measurement',
        'diabetes'  : 'FBG >= 7.0 mmol/L or known diagnosis [WHO/ADA] — STEPS real biomarker',
    },
    'training_data': {
        'dhs'  : 'Rwanda DHS 2019-20 — real haemoglobin, real BMI (women)',
        'steps': 'WHO STEPS Rwanda 2012-13 — real fasting glucose, real BMI (both sexes)',
    },
    'cv_scores': cv_results,
}
joblib.dump(bundle, BUNDLE_PATH)
print(f'Bundle saved → {BUNDLE_PATH}  ({os.path.getsize(BUNDLE_PATH)/1e6:.1f} MB)')
print(f'Data source  : {DATA_SOURCE}')

## Part 3 — HuggingFace Deployment

**Before running this cell:**  
Add-ons → Secrets → add a secret named `HF_TOKEN` with your HuggingFace write token.

In [ ]:
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
HF_USER  = 'JeanJabo'
SPACE_ID = f'{HF_USER}/nutrisense-api'

login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi()

files_to_push = {
    CKPT_PATH   : 'food_finetuned_model.pth',
    CLASSES_TXT : 'class_names.txt',
    BUNDLE_PATH : 'nutrisense_model.joblib',
}

print(f'Pushing to → https://huggingface.co/spaces/{SPACE_ID}')
for local, remote in files_to_push.items():
    if not os.path.exists(local):
        print(f'  ⚠️  {local} not found — skipping'); continue
    size_mb = os.path.getsize(local) / 1e6
    print(f'  Uploading {remote} ({size_mb:.1f} MB) ...')
    api.upload_file(
        path_or_fileobj=local,
        path_in_repo=remote,
        repo_id=SPACE_ID,
        repo_type='space',
        token=HF_TOKEN,
    )
    print(f'  ✅ {remote} uploaded')

print(f'\n🚀 Done → https://huggingface.co/spaces/{SPACE_ID}')

In [ ]:
# Final verification — all outputs
OUTPUT_FILES = {
    '10_feature_distributions.png' : 'Risk model feature distributions',
    '11_correlation_heatmap.png'   : 'Feature correlation matrix',
    '12_class_imbalance.png'       : 'Disease label distribution',
    '13_roc_pr_curves.png'         : 'ROC and Precision-Recall curves',
    '14_confusion_matrices.png'    : 'XGBoost confusion matrices (3 diseases)',
    '15_shap_importance.png'       : 'SHAP feature importance',
    '16_shap_waterfall.png'        : 'SHAP waterfall (individual explanation)',
    '17_cv_results.png'            : 'Cross-validation results table',
    'nutrisense_model.joblib'      : 'XGBoost risk model bundle',
}

print('=' * 62)
print('FINAL OUTPUT VERIFICATION')
print('=' * 62)
all_ok = True
for fname, desc in OUTPUT_FILES.items():
    path   = f'{SAVE_DIR}/{fname}'
    exists = os.path.exists(path)
    size   = f'{os.path.getsize(path)/1e6:.1f} MB' if exists else ''
    mark   = '✅' if exists else '❌'
    print(f'  {mark}  {desc:45s}  {size}')
    if not exists: all_ok = False

print()
print(f'Data source   : {DATA_SOURCE.upper()}')
print(f'XGBoost AUCs  : { {k: f"{v[\"roc_auc\"]:.3f}" for k, v in cv_results.items()} }')
print()
print('✅ ALL DONE' if all_ok else '⚠️  Some outputs missing — check above')